# Data Ingestion — COPY INTO, Auto Loader, Lakeflow Connect

## Reference domain

The bank has four very different ingestion problems, one per vertical. We'll use each as a worked example.

| Vertical | Source | Cadence | Format | Right answer |
|---|---|---|---|---|
| **Cards** | SFTP drop into a UC volume | Every 15 min | JSON, nested | **Auto Loader** |
| **Loans** | Operational PostgreSQL (loan origination system) | Nightly snapshot | Relational rows | **Lakeflow Connect managed connector** (SQL Server / Postgres) or **JDBC** |
| **Accounts** | Daily CSV exports landing in S3 | Daily, single drop | CSV | **`COPY INTO`** |
| **Payments** | Salesforce (case management) + Kafka (real-time UPI) | Continuous | API + stream | **Salesforce managed connector** + Auto Loader (streaming) |

By the end of this notebook you should be able to look at any new source the bank brings on and pick the right ingestion path in under thirty seconds.

## Three ingestion patterns — batch, incremental, streaming

Every source you'll ever ingest falls into one of three shapes.

**Batch.** The full dataset every time. "Here's all of yesterday's bank_accounts as a CSV." Simple, but expensive once the file is big and wasteful when only 1% changed.

**Incremental.** Only the new or changed records since the last load. The source emits a continuous trail of new files in object storage, or supports change-data-capture, or has a monotonic column you can high-watermark on. Most production lakehouse ingestion lives here.

**Streaming.** Records flow in continuously and the target should be updated continuously (within seconds to a few minutes). Same engine as incremental — Structured Streaming — but with a continuous trigger instead of a one-shot `availableNow`.

**The exam vocabulary you must own:**

- *Trigger.Once* — legacy name for *trigger.AvailableNow*. Process whatever's there, then stop. Used for scheduled "micro-batch on a cron" pipelines.
- *Trigger.ProcessingTime* — fire every N seconds/minutes. Used for continuous streaming.
- *Checkpoint location* — where Structured Streaming records the offsets it has already processed, so a restarted job picks up exactly where it left off (exactly-once semantics on the *write* path).

## `COPY INTO` — idempotent file ingestion in pure SQL

`COPY INTO` is a SQL statement that loads files from a cloud storage location into a Delta table, **remembers which files it has already loaded**, and skips them on the next run. That single property — idempotency — is what makes it the right answer for daily-batch file loads.

```sql
 COPY INTO fintech_dev.bronze.bank_accounts_raw
 FROM '/Volumes/fintech_dev/bronze/landing_zone/accounts/'
 FILEFORMAT = CSV
 FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
 COPY_OPTIONS  ('mergeSchema' = 'true');
```

**What's happening:**

- The target is a UC-governed Delta table (manage- or external-table doesn't matter to `COPY INTO`).
- The source path can be in any cloud object store the workspace has access to — ADLS, S3, or GCS — typically referenced through a UC **volume** or external location.
- After the first run, Delta records the loaded file paths. Re-running the same statement is a **no-op** for already-loaded files.
- `FORMAT_OPTIONS` are Spark reader options (the source format). `COPY_OPTIONS` are Delta writer options (the target table behaviour).

**When to pick `COPY INTO`:**

- A small to moderate number of files per run (hundreds, not millions).
- A predictable cadence — typically a Lakeflow Jobs task on a schedule.
- You want **declarative SQL** with no streaming machinery to debug.
- The source format is one of the supported formats (CSV, JSON, Avro, ORC, Parquet, XML, BINARYFILE).

## Auto Loader — `cloudFiles`, the workhorse

Auto Loader is a Structured Streaming source identified by the format string `cloudFiles`. It does one job and does it at any scale: **discover new files in a cloud directory and stream them into a Delta table**, with checkpointed exactly-once semantics on writes.

Same shape every time:

```python
 (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
  .writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("fintech_dev.bronze.card_transactions_raw"))
```

**Key options the exam tests:**

- `cloudFiles.format` — the underlying file format (`json`, `csv`, `parquet`, `avro`, `binaryFile`, `text`).
- `cloudFiles.schemaLocation` — where Auto Loader caches the inferred schema across runs.
- `cloudFiles.schemaEvolutionMode` — what to do when a new column appears (covered below).
- `checkpointLocation` (on the writer) — where Structured Streaming tracks processed offsets.
- `trigger(availableNow=True)` — process every file that exists *right now*, then stop. The modern replacement for `trigger.once`. Run it from a scheduled Lakeflow Jobs task for batch-like cadence on top of the streaming engine.

**Auto Loader wins over `COPY INTO` when:**

- The number of files per run is huge (thousands to millions).
- New files arrive *continuously*, not in a single daily drop.
- You want **schema evolution** baked in, not bolted on.
- You'd like to flip from scheduled (`availableNow`) to continuous later without rewriting the pipeline.

## Auto Loader — directory listing vs. file notification mode

Auto Loader has two ways to discover new files. The choice is a cost-and-scale trade.

```text
  Directory listing mode (default)        File notification mode
  ───────────────────────────────         ──────────────────────────────────
                                          ┌────────────┐
  ┌───────────┐ list files   ┌──────────┐ │ Cloud      │ object created event
  │ S3 bucket │ ────────────►│ Auto     │ │ storage    │ ──────────┐
  │ ADLS path │ every micro- │ Loader   │ │ S3/ADLS/GCS│           ▼
  │ GCS path  │ batch        │          │ └────────────┘   ┌──────────────┐
  └───────────┘              └──────────┘                  │ Pub/Sub queue│
                                                           │  (SQS/SNS,   │
                                                           │   Event Grid,│
                                                           │   Pub/Sub)   │
                                                           └──────┬───────┘
                                                                  ▼
                                                           ┌──────────────┐
                                                           │ Auto Loader  │
                                                           └──────────────┘
```

**Directory listing mode** — Auto Loader lists the source directory on each micro-batch and diffs against what it has already processed. The default. Works on any cloud, no extra cloud resources needed. Bottoms out at millions of files in a single directory, where listing becomes the bottleneck.

**File notification mode** — Auto Loader subscribes to a managed event queue (AWS SQS+SNS, Azure Event Grid+Queue Storage, GCP Pub/Sub). The cloud emits an event when each file lands; Auto Loader consumes the event and reads only that file — no listing required. Scales to billions of files. The trade: Auto Loader needs IAM permissions to *create and manage* the queue, and you pay queue costs.

**Rule of thumb the exam wants:**

- Few-to-millions of files in the directory? Directory listing is fine.
- Hundreds of millions to billions of files, or partition directories with high cardinality? Switch to **file notification**.

## Auto Loader schema enforcement and evolution — the four modes

Auto Loader infers the schema on the first batch (or uses a hint you supplied) and **caches it at `cloudFiles.schemaLocation`**. On every subsequent batch it compares the file's schema against the cache. What happens when there's a difference is controlled by `cloudFiles.schemaEvolutionMode`:

| Mode | Behaviour on a new column |
|---|---|
| **`addNewColumns`** (default) | Stream **fails** on the file containing the new column. Restart the stream — it picks up the new column, evolves the schema, and continues. Best for production: no silent drift, but easy recovery. |
| **`rescue`** | Stream continues. Unknown columns are captured into a single rescued-data column instead of being dropped. Nothing fails; nothing is silently lost. |
| **`failOnNewColumns`** | Stream fails permanently. You must explicitly evolve the schema yourself and restart. The strictest mode — for regulated data where a new column needs human review. |
| **`none`** | New columns are silently ignored. Use only when you fully control the upstream and a new column is a bug. |

**The rescued data column.** Auto Loader can be configured to add a `_rescued_data` column to every row. Anything that didn't fit the schema (a new column, a value that couldn't be cast) lands there as JSON. Combined with `rescue` mode, this means **nothing the source emits is ever silently dropped**. Critical for audit and reconciliation.

**Schema hints** — `cloudFiles.schemaHints = 'amount DECIMAL(18,2), is_flagged BOOLEAN'` overrides inference for the listed columns. Cheap way to pin types without writing a full schema.

## `COPY INTO` vs. Auto Loader — when each wins

Both load files from object storage into Delta. Both are idempotent (Auto Loader via checkpoint, `COPY INTO` via Delta-tracked load history). Pick by *operational shape*.

| | `COPY INTO` | Auto Loader |
|---|---|---|
| API | Pure SQL | DataFrame / Structured Streaming |
| Idempotency mechanism | Delta-tracked loaded-file list | Streaming checkpoint |
| Best for | Daily batch, small/medium file counts | Continuous or high-volume incremental |
| Schema evolution | Manual (`mergeSchema` per run) | Built-in (`schemaEvolutionMode`) |
| Throughput ceiling | Hundreds → low thousands of files per run | Millions+ via file notification |
| Latency | Run-cadence (cron) | Down to seconds with continuous trigger |
| Use case at the bank | Nightly `accounts` CSV drop | Sub-hourly `card_transactions` JSON drops |

**The single most common exam question pattern:**

- *"new files arrive continuously, schema may evolve"* → **Auto Loader**
- *"a single batch of files lands once a day, simple SQL preferred"* → **`COPY INTO`**

## Lakeflow Connect — the unified ingestion product

`COPY INTO` and Auto Loader handle the *files arriving in object storage* problem. But the bank also has data sitting in **operational systems** — PostgreSQL for loan origination, Salesforce for customer cases, NetSuite for finance. **Lakeflow Connect** is the Databricks-managed ingestion family that handles those.

Three tiers, from most-managed to most-flexible:

1. **Managed connectors** — fully Databricks-operated pipelines for SaaS and database sources. You configure source credentials and a target catalog; Databricks runs the connector, handles CDC, schema evolution, retries, and monitoring.
2. **Standard connectors** — the open ingestion building blocks: Auto Loader for files, Structured Streaming sources for Kafka / Kinesis / Event Hubs / Pub/Sub. You write the code; Databricks provides the runtime.
3. **Partner connectors** — Fivetran, Airbyte, dbt Labs, Hevo, Rivery, etc. Third-party ingestion tools with native Databricks integrations and joint billing.

All three land data into UC-governed Delta tables. The choice is about who operates the moving parts.

```text
        Sources                  Lakeflow Connect                  UC-governed Delta
 ┌────────────────┐                                              ┌──────────────────┐
 │ Salesforce     │ ──► managed connector ───────────────────►   │                  │
 │ Workday        │                                              │  fintech_dev     │
 │ ServiceNow     │                                              │   .bronze        │
 │ NetSuite       │                                              │     .salesforce  │
 │ Postgres / MS  │ ──► managed connector (DB ingestion / CDC)──►│     .postgres    │
 ├────────────────┤                                              │     .files       │
 │ S3 / ADLS /    │ ──► standard:  Auto Loader / COPY INTO ────► │     .kafka       │
 │ GCS files      │                                              │                  │
 │ Kafka / Kinesis│ ──► standard:  Structured Streaming ───────► │  (bronze layer)  │
 │  / Event Hubs  │                                              │                  │
 ├────────────────┤                                              │                  │
 │ Long tail SaaS │ ──► partner: Fivetran / Airbyte / dbt ─────► │                  │
 └────────────────┘                                              └──────────────────┘
```

## Managed connectors — SaaS and database CDC, operated by Databricks

A **managed connector** is a fully Databricks-run ingestion pipeline. You point it at a source (provide credentials, pick tables/objects), point it at a UC catalog, and pick a schedule. Databricks runs the connector on serverless compute, handles incremental ingestion (typically CDC where the source supports it), evolves the schema, retries failures, and exposes monitoring.

Currently shipping families include:

- **SaaS apps** — Salesforce, Workday, ServiceNow, NetSuite, Google Analytics, SharePoint.
- **Databases (with CDC)** — SQL Server, PostgreSQL, MySQL, Oracle, ServiceNow.
- **Event sources** — Kafka via the Confluent connector.

**The bank's use case.** The payments team uses Salesforce for case management. Customer service notes about disputed UPI transactions land in Salesforce. Instead of writing a custom REST poller that has to handle pagination, rate limits, and field changes, the team enables the **Salesforce managed connector**, picks which objects to sync (`Case`, `Contact`, `Account`), and points it at `fintech_dev.bronze.salesforce`. Databricks handles the rest, including incremental sync via Salesforce's CDC API.

**Why the exam likes managed connectors:**

- Lowest operational burden — no Python code, no JDBC fiddling, no retry logic.
- Native CDC where supported — only changed rows after the first snapshot.
- Built-in monitoring through the Lakeflow Jobs UI.
- Tables land in UC pre-governed, with lineage already in place.

## Partner connectors — Fivetran, Airbyte, and friends

**Partner connectors** are third-party ingestion tools that Databricks integrates with natively. They show up under *Partner Connect* in the workspace UI, sign-up and billing flow through Databricks, and they land tables directly into UC.

When to pick a partner over a managed connector:

- The source is in the long tail — Marketo, Stripe, Mixpanel, Zendesk, Shopify — where Fivetran or Airbyte has a connector and Lakeflow Connect does not yet.
- You're already standardised on a partner tool across other warehouses (Snowflake, BigQuery) and want one operational pattern for ingestion.
- You want a no-code UI for analysts to add new sources.

**The exam doesn't dig into specific partner tools.** It does expect you to recognise that Lakeflow Connect, partner connectors, and standard connectors are three *peer* options, and to pick *managed* when the source is supported.

## JDBC and ODBC — pulling from an operational database directly

When there's no managed connector for the database (or you need a one-off pull), you can read directly from a JDBC source in a notebook.

```python
 loans_df = (spark.read.format("jdbc")
     .option("url",      "jdbc:postgresql://loans-prod.bank.internal:5432/loans")
     .option("dbtable",  "public.loan_accounts")
     .option("user",      dbutils.secrets.get("loans", "username"))
     .option("password",  dbutils.secrets.get("loans", "password"))
     .option("driver",   "org.postgresql.Driver")
     .load())

 loans_df.write.mode("overwrite").saveAsTable("fintech_dev.bronze.loan_accounts_raw")
```

**What the exam wants you to know:**

- Credentials **must come from a secret scope**, never hardcoded. `dbutils.secrets.get(scope, key)` is the pattern.
- The JDBC driver JAR must be installed on the cluster (or use a DBR runtime that bundles it).
- For large pulls, use `partitionColumn`, `lowerBound`, `upperBound`, `numPartitions` to parallelise. Otherwise Spark reads the whole table on a single executor.
- This is a **pull** pattern — Spark drives it. Use it when the source can't push to object storage.
- It is typically wrapped in a **Lakeflow Jobs task** that runs nightly against a snapshot table, so the load is predictable.

## REST API ingestion — for sources that have neither a connector nor a database driver

For the long-tail source the bank uses — a fraud-scoring API from a third party, for instance — you may have to call REST yourself from a notebook.

The pattern is straightforward but the gotchas matter:

- **Paginate** — almost every API returns pages; loop until the response says there's no more.
- **Authenticate with secrets** — token in `dbutils.secrets`, never in the notebook.
- **Land raw JSON to a UC volume first**, *then* ingest with Auto Loader or `COPY INTO`. Don't pivot the API response straight into a Delta write inside the loop — you lose the audit copy and the ability to re-process.
- **Schedule with Lakeflow Jobs**, retries enabled.

**Two-stage pattern:**

```text
  Stage 1 (notebook task):   REST API ──► /Volumes/.../landing/fraud_scores/YYYY-MM-DD.json
  Stage 2 (notebook task):   Auto Loader ──► fintech_dev.bronze.fraud_scores
```

Treat REST as a **landing strategy**, not a target. The same Auto Loader / `COPY INTO` ingestion logic from earlier in this notebook then handles the *file-on-storage → Delta* hop, with all its idempotency and schema-evolution guarantees.

## Semi-structured and nested data — JSON into Delta

The card-transactions feed from the payment processor is nested JSON — each transaction is one object with nested `card`, `merchant`, and `auth` sub-objects. You have two options.

**Option 1 — Land it nested, flatten in silver.** This is the bronze-then-silver pattern and the one the exam rewards. Auto Loader infers a nested struct schema; the bronze table has columns of type `STRUCT<...>` or `ARRAY<STRUCT<...>>`. Silver flattens with dot-access, `explode()`, and `from_json()` where needed.

**Option 2 — Land it flat at the source.** Configure the upstream to emit flattened JSON. Cleaner downstream, but you give up the original shape (and replay-ability).

**Three PySpark / SQL primitives to memorise:**

- **Dot access** — `df.select("merchant.name", "merchant.category")`. Works on `STRUCT` fields.
- **`explode(arr)`** — turns an array column into one row per element. The exam pattern for unpacking line-items or events arrays.
- **`from_json(json_str, schema)`** — parse a stringified JSON column into a struct. Combined with `schema_of_json`, this is how you turn the `_rescued_data` column into typed fields.

**One option that often shows up in answers** — `spark.read.option("multiLine", "true").json(...)` — for JSON files where the entire file is one object spread across lines, rather than newline-delimited JSON. Auto Loader supports `cloudFiles.format = json` with `multiLine` as a format option.

## Lakehouse Federation — when *not* to ingest at all

Sometimes the right answer to *"how should we ingest this?"* is **don't**. **Lakehouse Federation** lets Unity Catalog register foreign databases (PostgreSQL, MySQL, SQL Server, Snowflake, Redshift, BigQuery) as UC catalogs and **query them in place** through Databricks. No data movement, no pipeline.

```sql
 SELECT *
 FROM federated_postgres.public.loan_accounts
 WHERE status = 'defaulted';
```

**Use federation when:**

- The source data is small or you query it rarely (the cost of ingestion outweighs querying live).
- Latency matters more than throughput — you need the *current* row, not yesterday's snapshot.
- Compliance forbids copying the data into the lakehouse.

**Don't use federation when:**

- The data is hot for the platform — analysts query it daily. Ingest it instead so the source DB isn't hammered.
- You need joins across multiple sources at scale. Each federated query reaches across the network.
- You need historical / time-travel semantics. Federation reads the *current* source state.

Federation is one row in the decision sheet below — but a row the exam will plant in distractors.

## The decision sheet — pick the right ingestion path

Use this as the cheat sheet for any new source the bank brings on.

| Source shape | Pick |
|---|---|
| Daily CSV/Parquet drop, few files, simple SQL preferred | **`COPY INTO`** |
| Files arrive continuously, possibly millions, schema may evolve | **Auto Loader** (`cloudFiles`) |
| Auto Loader at billion-file scale, listing too slow | **Auto Loader + file notification mode** |
| SaaS app with a managed connector (Salesforce, Workday, ServiceNow, NetSuite, SharePoint) | **Lakeflow Connect managed connector** |
| Relational DB with CDC (SQL Server, Postgres, MySQL, Oracle) | **Lakeflow Connect managed DB connector** |
| Kafka / Kinesis / Event Hubs / Pub/Sub | **Structured Streaming** (standard connector) |
| Long-tail SaaS not in managed connectors | **Partner connector** (Fivetran / Airbyte) |
| Operational DB, no managed connector, nightly snapshot | **JDBC** in a Lakeflow Jobs task |
| Custom REST API | Land raw JSON to a UC volume → Auto Loader |
| Small / cold source, query rarely, no copies allowed | **Lakehouse Federation** (don't ingest) |

Two pivot questions that resolve most exam scenarios:

1. *Is the source files in object storage, or a system with its own API?* → files → `COPY INTO` / Auto Loader; API → managed connector / JDBC / federation.
2. *Do I need every change as it happens (streaming), every file as it lands (incremental), or once a day (batch)?* → drives trigger choice and whether you reach for streaming machinery at all.

## Hands-on — ingesting the bank's four verticals

The cells below assume the `fintech_dev` catalog and the `bronze` schema you created in notebook 02 already exist, plus the `bronze.landing_zone` volume.

Run this in a **Databricks notebook attached to a cluster** (DBR 14.x+). For the JDBC cell, you'll need a reachable Postgres — substitute your own connection or skip the cell.

In [ ]:
# Set the working catalog/schema so we don't repeat fintech_dev.bronze everywhere.
spark.sql("USE CATALOG fintech_dev")
spark.sql("USE SCHEMA bronze")

# Subdirectories inside the landing volume for each vertical's raw drops.
LANDING = "/Volumes/fintech_dev/bronze/landing_zone"
for sub in ("accounts", "cards", "fraud_scores"):
    dbutils.fs.mkdirs(f"{LANDING}/{sub}")

# Schema + checkpoint directories live in the same volume — UC-governed.
for sub in ("_schemas/cards", "_checkpoints/cards"):
    dbutils.fs.mkdirs(f"{LANDING}/{sub}")

In [ ]:
# Seed two daily CSV drops for bank_accounts. In production these arrive from the source SFTP.
day1 = "account_id,customer_id,account_type,balance,opened_at,status\n" \
       "A001,C001,savings,15000.00,2022-01-10T00:00:00,active\n" \
       "A002,C002,current,82000.00,2021-06-05T00:00:00,active\n"

day2 = "account_id,customer_id,account_type,balance,opened_at,status\n" \
       "A003,C003,fixed_deposit,500000.00,2023-03-20T00:00:00,active\n" \
       "A004,C001,savings,17000.00,2024-01-15T00:00:00,active\n"

dbutils.fs.put(f"{LANDING}/accounts/2024-01-05.csv", day1, overwrite=True)
dbutils.fs.put(f"{LANDING}/accounts/2024-01-06.csv", day2, overwrite=True)

dbutils.fs.ls(f"{LANDING}/accounts")

In [ ]:
# Daily-drop, simple SQL — exactly what COPY INTO is for.
spark.sql("""
    CREATE TABLE IF NOT EXISTS bank_accounts_raw (
        account_id   STRING,
        customer_id  STRING,
        account_type STRING,
        balance      DECIMAL(18,2),
        opened_at    TIMESTAMP,
        status       STRING
    ) USING DELTA
""")

spark.sql(f"""
    COPY INTO bank_accounts_raw
    FROM '{LANDING}/accounts/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""").show(truncate=False)

spark.sql("SELECT COUNT(*) AS rows, MAX(opened_at) AS latest_opened FROM bank_accounts_raw").show()

In [ ]:
# Re-running COPY INTO is a NO-OP for already-loaded files. Proof: same row count, num_inserted_rows = 0.
spark.sql(f"""
    COPY INTO bank_accounts_raw
    FROM '{LANDING}/accounts/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
""").show(truncate=False)

In [ ]:
# Seed two nested-JSON drops for card_transactions — newline-delimited, one transaction per line.
# The 10:00 drop has the original schema. The 10:15 drop adds a new field 'merchant.country'.
drop_10_00 = (
    '{"transaction_id":"T100","customer_id":"C001","amount":1500.00,'
    '"merchant":{"name":"BigBasket","category":"grocery"},"transaction_at":"2024-01-05T10:30:00"}\n'
    '{"transaction_id":"T101","customer_id":"C002","amount":8200.00,'
    '"merchant":{"name":"MakeMyTrip","category":"travel"},"transaction_at":"2024-01-06T14:10:00"}\n'
)
drop_10_15 = (
    '{"transaction_id":"T102","customer_id":"C003","amount":3000.00,'
    '"merchant":{"name":"Shell","category":"fuel","country":"IN"},"transaction_at":"2024-01-07T09:00:00"}\n'
)
dbutils.fs.put(f"{LANDING}/cards/2024-01-05_10-00.json", drop_10_00, overwrite=True)
dbutils.fs.put(f"{LANDING}/cards/2024-01-05_10-15.json", drop_10_15, overwrite=True)

In [ ]:
# Auto Loader, JSON, schemaEvolutionMode = rescue.
# trigger(availableNow=True) means: process every file present now, then stop — perfect for
# a Lakeflow Jobs task scheduled every 15 minutes.
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation",       f"{LANDING}/_schemas/cards")
    .option("cloudFiles.schemaEvolutionMode",  "rescue")
    .option("cloudFiles.inferColumnTypes",     "true")
    .load(f"{LANDING}/cards")
  .writeStream
    .option("checkpointLocation", f"{LANDING}/_checkpoints/cards")
    .trigger(availableNow=True)
    .toTable("card_transactions_raw")
    .awaitTermination())

spark.sql("SELECT * FROM card_transactions_raw").show(truncate=False)

In [ ]:
# DESCRIBE shows the inferred nested schema PLUS the _rescued_data column.
# Anything Auto Loader couldn't fit lands there as JSON — nothing is silently lost.
spark.sql("DESCRIBE TABLE card_transactions_raw").show(truncate=False)

In [ ]:
# Flatten the nested struct — bronze keeps it nested; silver flattens.
# This is the snippet you'd write into the bronze-to-silver transformation in notebook 04.
spark.sql("""
    SELECT transaction_id,
           customer_id,
           amount,
           merchant.name      AS merchant_name,
           merchant.category  AS merchant_category,
           transaction_at
    FROM card_transactions_raw
    ORDER BY transaction_at
""").show(truncate=False)

In [ ]:
# JDBC — the production shape. Commented out because it needs a reachable Postgres.
# Read it as the canonical recipe.
#
# loans_df = (spark.read.format("jdbc")
#     .option("url",      "jdbc:postgresql://loans-prod.bank.internal:5432/loans")
#     .option("dbtable",  "public.loan_accounts")
#     .option("user",      dbutils.secrets.get("loans", "username"))
#     .option("password",  dbutils.secrets.get("loans", "password"))
#     .option("driver",   "org.postgresql.Driver")
#     # For large tables — parallelise:
#     .option("partitionColumn", "loan_id")
#     .option("lowerBound",      "1")
#     .option("upperBound",      "10000000")
#     .option("numPartitions",   "8")
#     .load())
#
# loans_df.write.mode("overwrite").saveAsTable("loan_accounts_raw")

print("JDBC pattern shown above — wire it to your Postgres and uncomment.")

## Section 2 self-check

Five exam-style questions. Answers at the bottom.

**1.** A new vendor sends the bank a single CSV file once a day to an S3 path. The data engineer wants a simple SQL-only ingestion that is idempotent and runs from a scheduled Lakeflow Jobs task. Which ingestion method is the best fit?

- A. Auto Loader with `availableNow`
- B. `COPY INTO`
- C. Lakehouse Federation
- D. JDBC pull

**2.** The card processor drops nested-JSON files into an ADLS landing path every 30 seconds. Some fields may be added by the source over time. Which ingestion path matches?

- A. `COPY INTO` once a day
- B. Auto Loader with `cloudFiles.schemaEvolutionMode = addNewColumns`
- C. A nightly JDBC pull from the processor's DB
- D. Lakehouse Federation

**3.** The bank uses Salesforce for case management and wants to ingest `Case`, `Contact`, and `Account` into UC-governed Delta tables with minimal operational overhead. Which is the best choice?

- A. Write a Python notebook that calls the Salesforce REST API on a schedule
- B. Use a Lakeflow Connect managed Salesforce connector
- C. Mount the Salesforce data lake to DBFS
- D. Use `COPY INTO` against the Salesforce SOAP endpoint

**4.** Auto Loader is processing billions of files in a deeply nested S3 prefix. Directory listing has become the bottleneck. Which option fixes the throughput issue?

- A. Switch to `COPY INTO`
- B. Enable `cloudFiles.useNotifications` (file notification mode)
- C. Increase `spark.sql.shuffle.partitions`
- D. Set `cloudFiles.schemaEvolutionMode = none`

**5.** A small dimension table sits in a Postgres database and the bank queries it once a week from notebooks. Compliance forbids copying it into the lakehouse. Which approach fits?

- A. Nightly JDBC snapshot
- B. Auto Loader
- C. Lakehouse Federation
- D. `COPY INTO`

<details><summary>Show answers</summary>

1. **B** — `COPY INTO` is idempotent SQL ingestion for daily file drops.
2. **B** — Auto Loader handles continuous file arrival with schema evolution.
3. **B** — managed connectors take the operational load off the team and land tables directly in UC.
4. **B** — file notification mode bypasses directory listing entirely; ingestion scales to billions of files.
5. **C** — Lakehouse Federation queries the source in place, with no copy.

</details>